In [0]:
%sql
-- INGESTA
INSERT OVERWRITE vitivinicola_catalog.silver.productos
SELECT 
    id,
    titulo,
    bodega,
    tipo,           
    tipo_inferido,  
    COALESCE(tipo, tipo_inferido) AS categoria,    
    tags,
    precio,
    precio_tachado,
    stock,
    publicado,
    tienda,
    pais,
    moneda,
    _ingestion_ts,
    _source_file
FROM (
  SELECT 
    CASE 
      WHEN id RLIKE '^[0-9]+$' THEN CAST(id AS BIGINT)
      ELSE NULL
    END AS id,
    titulo,
    CASE 
      WHEN bodega = 'Château Mouton-Rothschild' THEN 'Château Mouton Rothschild'
      WHEN bodega LIKE '%Tenuta%Ornellaia%' OR bodega LIKE '%Tenuta%dell%Ornellaia%' 
      THEN 'Tenuta dell''Ornellaia'      
      WHEN bodega IN ('Paco y Lola', 'Paco & Lola') THEN 'Paco & Lola'
      WHEN bodega ='Bodega Norton' THEN 'Norton'
      WHEN bodega = 'Tienda Susana Balbo' THEN 'Susana Balbo'
      WHEN bodega LIKE '%Arzuaga%' THEN 'Arzuaga Navarro'
      ELSE bodega
    END AS bodega,
    CASE 
      WHEN tipo LIKE 'Vino%' OR tipo LIKE 'vino%' OR tipo LIKE 'Wine' THEN 'Vino'
      WHEN tipo LIKE 'Espumante%' THEN 'Espumante'
      WHEN tipo LIKE 'Grappa' OR tipo LIKE 'Pisco' OR tipo LIKE 'Vodka' OR tipo LIKE 'Spirits' THEN 'Destilado'
      WHEN tipo LIKE 'Aceite%' OR tipo LIKE 'Mieles' THEN 'Delicatessen'
      WHEN tipo LIKE 'Cajas Mix%' OR tipo LIKE 'Bebidas alcohólicas%' THEN 'Otros'
    END AS tipo,
    CASE 
      WHEN tags LIKE '%Vinos%' OR tags LIKE '%Tintos%'
        OR tags LIKE '%vinos%' OR tags LIKE '%tintos%'
        OR tags LIKE '%Blancos%' OR tags LIKE '%Rosado%' 
        OR tags LIKE '%Malbec%' OR tags LIKE '%Cabernet%' 
        OR tags LIKE '%Crios%' OR tags LIKE '%denuestracava%'
        THEN 'Vino'
      WHEN tags LIKE '%Espumosos%'  THEN 'Espumante'
      WHEN tags LIKE '%Vermouth%' OR tags LIKE '%Vermut%' OR tags LIKE '%Vermú%'  
        THEN 'Vermut'
      WHEN tags LIKE '%Destilados%' OR tags LIKE '%Whisky%'  
        OR tags LIKE '%Ron%' OR tags LIKE '%Mezcal%'  
        OR tags LIKE '%grappa%' OR tags LIKE '%Digestivo%'
        THEN 'Destilado'
      WHEN tags LIKE '%Catas%' OR tags LIKE '%Cursos%' THEN 'Sin clasificar'
      WHEN tags LIKE '%elicatessen%' OR tags LIKE '%eilicatessen%' THEN 'Delicatessen'
      WHEN titulo LIKE '%ermouth%' OR titulo LIKE '%ermut%'  THEN 'Vermut'
      ELSE 'Sin clasificar'
    END AS tipo_inferido,
    tags,
    CASE
      WHEN precio RLIKE '^[0-9]+(\.[0-9]+)?$' THEN CAST(precio AS DOUBLE)
      ELSE NULL
    END AS precio, 
    CASE
      WHEN precio_tachado RLIKE '^[0-9]+(\.[0-9]+)?$' THEN CAST(precio_tachado AS DOUBLE)
      ELSE NULL
    END AS precio_tachado,
    TRY_CAST(stock AS BOOLEAN) AS stock,
    publicado,
    CASE 
      WHEN _source_file LIKE '%balbo%' THEN 'Susana Balbo'
      WHEN _source_file LIKE '%norton%' THEN 'Norton'
      WHEN _source_file LIKE '%ocio%' THEN 'Ocio Wine'
      WHEN _source_file LIKE '%miro%' THEN 'Exclusivas Miro'
      WHEN _source_file LIKE '%barrica%' THEN 'La Barrica'
    END AS tienda,
    pais,
    moneda,
    _ingestion_ts, 
    _source_file, 
  -- LÓGICA DE DETECCIÓN DE DUPLICADOS
    ROW_NUMBER() OVER(PARTITION BY CASE WHEN _source_file LIKE '%balbo%' THEN 'Susana Balbo' WHEN _source_file LIKE '%norton%' THEN 'Norton' WHEN _source_file LIKE '%ocio%' THEN 'Ocio Wine' WHEN _source_file LIKE '%miro%' THEN 'Exclusivas Miro' WHEN _source_file LIKE '%barrica%' THEN 'La Barrica' END, titulo ORDER BY CAST(precio AS DOUBLE) ASC) AS row_num
  FROM vitivinicola_catalog.bronze.productos
  WHERE pais IN ('Argentina','USA','Uruguay','España') 
    AND precio RLIKE '^[0-9]+(\.[0-9]+)?$' 
    AND id RLIKE '^[0-9]+$'


)
WHERE COALESCE(tipo, tipo_inferido) IN ('Vino', 'Espumante', 'Vermut')
AND row_num = 1 
;

-- Se identificaron registros con igual título dentro de una misma tienda y diferencias de precio asociadas a 
-- distintas presentaciones comerciales. Para evitar duplicidades en el análisis se conservó únicamente el 
-- registro de menor precio por combinación tienda-título.